<a href="https://colab.research.google.com/github/jcmachicao/MachineLearningAvanzado_UC_2025/blob/main/U1_AutoML_sklearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Using AutoGluon for AutoML

First, we need to install `AutoGluon`. This might take a few minutes as it has several dependencies.

In [6]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

Now, let's import the necessary libraries and prepare some sample data using `sklearn.datasets.load_digits`, similar to your original task. AutoGluon's `TabularPredictor` works well with Pandas DataFrames.

In [7]:
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor

# Load the digits dataset
digits = load_digits()
X, y = digits.data, digits.target

# Combine X and y into a single DataFrame, as AutoGluon expects this format
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

# Split data into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
display(train_df.head())

Training data shape: (1437, 65)
Test data shape: (360, 65)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,target
1734,0.0,0.0,3.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,11.0,16.0,13.0,4.0,0.0,6
855,0.0,0.0,9.0,9.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,6.0,16.0,14.0,3.0,0.0,0.0,0
1642,0.0,0.0,0.0,10.0,13.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,11.0,13.0,6.0,0.0,0.0,0
175,0.0,1.0,10.0,16.0,16.0,11.0,0.0,0.0,0.0,5.0,...,0.0,0.0,1.0,15.0,14.0,11.0,4.0,0.0,0.0,3
925,0.0,0.0,6.0,14.0,13.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,4.0,15.0,16.0,9.0,0.0,0.0,0


Next, we initialize and train the `TabularPredictor`. We specify the `label` (target column), and `time_limit` for how long AutoGluon should search for the best model. AutoGluon automatically handles preprocessing, feature engineering, and model selection/ensembling.

In [8]:
# Define the label column
label = 'target'

# Initialize the TabularPredictor
# We'll set a short time_limit for this example to run quickly.
# For better performance, increase time_limit.
predictor = TabularPredictor(label=label, eval_metric='accuracy', path='AutogluonModels').fit(
    train_data=train_df,
    time_limit=120, # seconds
    presets='medium' # Can be 'best_quality', 'high_quality', 'medium_quality', 'fast_inference'
)

# Display the leaderboard of trained models
leaderboard = predictor.leaderboard(test_df, silent=True)
display(leaderboard)

Preset alias specified: 'medium' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       11.06 GB / 12.67 GB (87.3%)
Disk Space Avail:   75.37 GB / 107.72 GB (70.0%)
Presets specified: ['medium']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/content/AutogluonModels"
Train Data Rows:    1437
Train Data Columns: 64
Label Column:       target
AutoGluon infers your prediction problem is: 'multiclass' (because dtype of label-column == int, but few unique label-values observed).
	10 unique label values:  [np.int64(6), np.int64(0), np.int64(3), np.int64(5), np.int64(4), np.i

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBMXT,0.986111,0.986111,accuracy,0.097793,0.049646,4.311886,0.097793,0.049646,4.311886,1,True,2
1,WeightedEnsemble_L2,0.986111,0.986111,accuracy,0.102368,0.050497,4.387501,0.004575,0.000851,0.075616,2,True,12
2,ExtraTreesEntr,0.983333,0.979167,accuracy,0.109798,0.075958,0.875377,0.109798,0.075958,0.875377,1,True,8
3,CatBoost,0.980556,0.979167,accuracy,0.008287,0.002924,8.105426,0.008287,0.002924,8.105426,1,True,6
4,LightGBM,0.980556,0.979167,accuracy,0.080158,0.041432,1.736216,0.080158,0.041432,1.736216,1,True,3
5,NeuralNetFastAI,0.977778,0.979167,accuracy,0.016402,0.014028,4.006856,0.016402,0.014028,4.006856,1,True,1
6,ExtraTreesGini,0.972222,0.979167,accuracy,0.097277,0.077220,0.867695,0.097277,0.077220,0.867695,1,True,7
7,RandomForestEntr,0.972222,0.979167,accuracy,0.097292,0.076375,0.887896,0.097292,0.076375,0.887896,1,True,5
8,RandomForestGini,0.969444,0.982639,accuracy,0.099992,0.077063,1.110359,0.099992,0.077063,1.110359,1,True,4
9,NeuralNetTorch,0.966667,0.979167,accuracy,0.024940,0.019600,9.793981,0.024940,0.019600,9.793981,1,True,10


Finally, we can use the trained `predictor` to make predictions on the test set and evaluate its performance.

In [9]:
# Make predictions on the test data
# We only pass the features from the test_df to predict
y_pred = predictor.predict(test_df.drop(columns=[label]))

# Get the true labels from the test data
y_true = test_df[label]

# Evaluate the predictions
accuracy = predictor.evaluate(test_df)['accuracy']
print(f"AutoGluon Test Accuracy: {accuracy:.4f}")

AutoGluon Test Accuracy: 0.9861
